In [ ]:
###########################################
# Unified Encoder Factory
###########################################
import torch
import torch.nn as nn
from scipy.io import wavfile
from scipy import signal
import numpy as np
import glob
import tempfile
import datetime, os, json, sys

import matplotlib.pyplot as plt
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append('/om2/user/bjmedina/')

import DistanceMemoryModel

from encoders import (
    Kell2018Encoder,
    AudioTextureEncoderPCA,
)

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params

from sklearn.decomposition import PCA

sys.path.append('/om/user/jmhicks/projects/TextureSimilarity/code/')
import texture_similarity.utils as ts


import scipy.signal
import scipy.signal.windows

if not hasattr(scipy.signal, "kaiser"):
    scipy.signal.kaiser = scipy.signal.windows.kaiser


def make_encoder(cfg):
    """
    Create an encoder model based on cfg.
    cfg fields:
        - encoder_type: 'kell2018', 'resnet50', 'texture_pca'
        - model_name, layer, task (for neural nets)
        - statistics_dict, pc_dims, model_params (for PCA texture model)
        - sr, rms_level, duration, device
    """
    etype = cfg['encoder_type']
    
    if etype == 'kell2018':
        return Kell2018Encoder(
            model_name=cfg['model_name'],
            layer=cfg['layer'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )
    
    elif etype == 'resnet50':
        return ResNet50Encoder(
            model_name=cfg['model_name'],
            layer=cfg['layer'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )
    
    elif etype == 'texture_pca':
        return AudioTextureEncoderPCA(
            statistics_dict=cfg['statistics_dict'],
            pc_dims=cfg['pc_dims'],
            model_params=cfg['model_params'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )

    else:
        raise ValueError(f"Unknown encoder_type: {etype}")

def encode_batch(encoder, sound_list):
    """
    Encode a list of raw audio arrays and return a (N, D) tensor.
    Assumes encoder(sound) returns a tensor or something with .reshape().
    """
    feats = []
    for s in sound_list:
        out = encoder(s)
        if isinstance(out, dict):
            out = out["embedding"]           # modify if your dict key differs
        feats.append(out.reshape(-1))         # flatten everything

    return torch.stack(feats, dim=0)

def make_model_save_dir(base_dir, model_info, which_task="test"):
    date_tag = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    enc = model_info.get("encoder_info", {})
    encoder_tag = "_".join([
        enc.get("encoder_type", "enc"),
        enc.get("model_name", "NA"),
        enc.get("layer", "NA")
    ])

    parts = [
        model_info.get('model_name', 'unknown'),
        model_info.get('metric', 'metric'),
        model_info.get('noise_mode', 'noise'),
        encoder_tag,
        f"rate{model_info.get('rate', 'NA')}",
        model_info.get('run_id', date_tag)
    ]

    subdir = "_".join(str(p) for p in parts if p)
    save_dir = os.path.join(base_dir, subdir)
    os.makedirs(save_dir, exist_ok=True)

    # write metadata
    info_path = os.path.join(save_dir, f"{which_task}_info.json")
    if not os.path.exists(info_path):
        with open(info_path, "w") as f:
            json.dump(model_info, f, indent=2)
    return save_dir

def encoder_metadata(cfg):
    """
    Extract key encoder fields for saving.
    """
    etype = cfg['encoder_type']
    base = {
        "encoder_type": etype,
        "sr": cfg.get("sr", 20000),
        "duration": cfg.get("duration", 2.0),
        "rms_level": cfg.get("rms_level", 0.05),
    }

    if etype in ["kell2018", "resnet50"]:
        base.update({
            "model_name": cfg['model_name'],
            "task": cfg.get("task", None),
            "layer": cfg['layer']
        })

    if etype == "texture_pca":
        base.update({
            "pc_dims": cfg['pc_dims'],
            "statistics_source": "texture_statistics_set",
        })

    return base

In [ ]:
###########################################
# Example Usage
###########################################
cfg = {
    "encoder_type": "kell2018",
    "model_name": "kell2018",
    "layer": "relu3",
    "task": "word_speaker_audioset",
    "sr": 20000,
    "duration": 2.0,
    "rms_level": 0.05,
    "device": "cuda"
}

sys.path.append(f'/om2/user/bjmedina/models/cochdnn/model_directories/{cfg['model_name']}_{cfg['task']}/')
print(f'/om2/user/bjmedina/models/cochdnn/model_directories/{cfg['model_name']}_{cfg['task']}/')

encoder = make_encoder(cfg)

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")

# encode some sounds
X = encode_batch(encoder, ALL_SOUNDS[:100])
print("Shape:", X.shape)

# build metadata
model_info = {
    "model_name": "DistanceMemoryModel",
    "metric": "cosine",
    "noise_mode": "exp-decay",
    "rate": 0.05,
    "encoder_info": encoder_metadata(cfg),
    "run_id": "runA01"
}

save_dir = make_model_save_dir("/om2/user/bjmedina/auditory-memory/memory/figures/test/", model_info)
print("Saved to:", save_dir)